# Document Chunking Quality Benchmark

Evaluates 4 chunking strategies across AIF example documents.

**Why chunking quality matters for LLM understanding:** When documents are split into chunks for
retrieval-augmented generation or multi-step reasoning, the quality of those chunks directly affects
how well LLMs understand and follow the content. AIF's typed semantic blocks (`@claim`, `@evidence`,
`@step`, `@verify`) provide natural chunking boundaries that preserve meaning:

- **Typed blocks give LLMs structural cues** --- a chunk beginning with `@step[order=3]` tells the
  model exactly what kind of content follows, enabling more accurate instruction following.
- **Semantic boundaries prevent mid-instruction splits** --- poorly split chunks that break between
  a `@step` and its `@verify` force the model to reason without validation criteria.
- **Self-contained chunks reduce hallucination** --- each chunk with clear typed blocks gives the
  model a complete reasoning unit instead of a fragment.

This benchmark is **fully deterministic** --- it uses the `aif chunk split` CLI command and requires no API key.

## Setup

In [1]:
import subprocess
import json
import math
import os
import statistics
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath("")).parent.parent
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/liqunchen/Claude_Project/token_efficient


## Strategies Overview

AIF supports 4 document chunking strategies, each with different trade-offs:

| Strategy | How it splits | Best for |
|----------|--------------|----------|
| **Section** | Splits on document section boundaries (`@section` blocks). Each section becomes its own chunk. | Documents with clear hierarchical structure --- each chunk maps to a logical topic. |
| **Token Budget** | Greedily fills chunks up to a target token limit (default 2048). Respects block boundaries. | Fitting chunks into LLM context windows --- guarantees no chunk exceeds the budget. |
| **Semantic** | Splits on semantic block boundaries (`@claim`, `@evidence`, `@step`, etc.). Groups related typed blocks together. | Preserving meaning --- keeps semantic units intact so LLMs can reason about typed content. |
| **Fixed Blocks** | Splits every N blocks (default ~5) regardless of content. | Uniform chunk sizes --- simple and predictable, but may cut across semantic boundaries. |

### Metrics

- **Self-Containment**: Fraction of chunks that have a title (a titled chunk gives the LLM immediate context about what it is reading).
- **Size CV** (Coefficient of Variation): How uniform chunk sizes are. Lower is better --- wildly varying chunk sizes waste context window space on tiny chunks and may overflow on large ones.
- **Avg Blocks/Chunk**: Average number of AIF blocks per chunk. More blocks per chunk means more context per retrieval hit.
- **Budget Compliance** (token-budget only): Fraction of chunks within 10% of the target token limit.

## Configuration

In [2]:
STRATEGIES = ["section", "token-budget", "semantic", "fixed-blocks"]
MAX_TOKENS = 2048

# Discover all example .aif files
examples_dir = PROJECT_ROOT / "examples"
aif_files = sorted(examples_dir.glob("**/*.aif"))
print(f"Found {len(aif_files)} .aif files:")
for f in aif_files:
    print(f"  {f.relative_to(PROJECT_ROOT)}")

Found 40 .aif files:
  examples/documents/simple.aif
  examples/documents/wiki_article.aif
  examples/migrations/migration_eslint_flat_config.aif
  examples/migrations/migration_nextjs_13_to_15.aif
  examples/migrations/migration_typescript_strict.aif
  examples/rich-content/climate_data.aif
  examples/skills/api-debugging.aif
  examples/skills/base-debugging.aif
  examples/skills/claude-opus-4-5-migration.aif
  examples/skills/code_review.aif
  examples/skills/commit-commands.aif
  examples/skills/feature-dev.aif
  examples/skills/frontend-design.aif
  examples/skills/gstack/gstack-autoplan.aif
  examples/skills/gstack/gstack-benchmark.aif
  examples/skills/gstack/gstack-canary.aif
  examples/skills/gstack/gstack-careful.aif
  examples/skills/gstack/gstack-checkpoint.aif
  examples/skills/gstack/gstack-codex.aif
  examples/skills/gstack/gstack-freeze.aif
  examples/skills/gstack/gstack-gstack-upgrade.aif
  examples/skills/gstack/gstack-guard.aif
  examples/skills/gstack/gstack-investi

## Chunking Functions

In [3]:
def run_cli(args: list[str]) -> str:
    """Run aif-cli and return stdout."""
    result = subprocess.run(
        ["cargo", "run", "-p", "aif-cli", "--"] + args,
        capture_output=True, text=True, cwd=str(PROJECT_ROOT),
    )
    if result.returncode != 0:
        print(f"  CLI error: {result.stderr.strip()[:200]}")
        return ""
    return result.stdout


def chunk_document(aif_file: str, strategy: str, max_tokens: int = MAX_TOKENS) -> list[dict]:
    """Chunk a document and return parsed chunk metadata."""
    args = ["chunk", "split", aif_file, "--strategy", strategy]
    if strategy == "token-budget":
        args += ["--max-tokens", str(max_tokens)]
    output = run_cli(args)
    if not output:
        return []

    chunks = []
    for line in output.strip().split("\n"):
        if not line.strip():
            continue
        # Format: "chunk_id | blocks: N | tokens: ~T | title: ..."
        parts = line.split("|")
        if len(parts) >= 4:
            chunk_id = parts[0].strip()
            blocks = int(parts[1].strip().replace("blocks: ", ""))
            tokens = int(parts[2].strip().replace("tokens: ~", ""))
            title = parts[3].strip().replace("title: ", "")
            chunks.append({
                "id": chunk_id,
                "blocks": blocks,
                "tokens": tokens,
                "title": title if title != "(none)" else None,
            })
    return chunks


def evaluate_chunking(aif_file: str, strategy: str, max_tokens: int = MAX_TOKENS) -> dict:
    """Evaluate a chunking strategy on a single document. Returns metrics dict."""
    chunks = chunk_document(aif_file, strategy, max_tokens)
    if not chunks:
        return {"error": "no chunks produced", "file": os.path.basename(aif_file), "strategy": strategy}

    n = len(chunks)
    tokens_list = [c["tokens"] for c in chunks]
    blocks_list = [c["blocks"] for c in chunks]

    # Self-containment: fraction of chunks with a title
    titled = sum(1 for c in chunks if c["title"])
    self_containment = titled / n if n > 0 else 0

    # Token budget compliance (token-budget strategy only)
    if strategy == "token-budget":
        within_budget = sum(1 for t in tokens_list if t <= max_tokens * 1.1)
        budget_compliance = within_budget / n if n > 0 else 0
    else:
        budget_compliance = None

    # Size coefficient of variation
    mean_tokens = sum(tokens_list) / n if n > 0 else 0
    if mean_tokens > 0 and n > 1:
        variance = sum((t - mean_tokens) ** 2 for t in tokens_list) / (n - 1)
        cv = math.sqrt(variance) / mean_tokens
    else:
        cv = 0.0

    # Average blocks per chunk
    avg_blocks = sum(blocks_list) / n if n > 0 else 0

    return {
        "file": os.path.basename(aif_file),
        "strategy": strategy,
        "num_chunks": n,
        "total_tokens": sum(tokens_list),
        "mean_tokens": round(mean_tokens, 1),
        "min_tokens": min(tokens_list) if tokens_list else 0,
        "max_tokens": max(tokens_list) if tokens_list else 0,
        "self_containment": round(self_containment, 3),
        "budget_compliance": round(budget_compliance, 3) if budget_compliance is not None else None,
        "size_cv": round(cv, 3),
        "avg_blocks_per_chunk": round(avg_blocks, 1),
    }

## Run Benchmark

Run all 4 strategies against every discovered `.aif` file. This is deterministic and requires no API key.

In [4]:
all_results = []
file_count = len(aif_files)
total_runs = file_count * len(STRATEGIES)
completed = 0

for aif_file in aif_files:
    rel = aif_file.relative_to(PROJECT_ROOT)
    print(f"\n--- {rel} ---")
    for strategy in STRATEGIES:
        result = evaluate_chunking(str(aif_file), strategy)
        all_results.append(result)
        completed += 1
        if "error" in result:
            print(f"  [{completed}/{total_runs}] {strategy:15s}  ERROR: {result['error']}")
        else:
            sc_pct = f"{result['self_containment']:.0%}"
            print(
                f"  [{completed}/{total_runs}] {strategy:15s}  "
                f"chunks={result['num_chunks']:2d}  "
                f"avg_tok={result['mean_tokens']:7.1f}  "
                f"self_cont={sc_pct:>4s}  "
                f"cv={result['size_cv']:.2f}"
            )

print(f"\nDone. {len(all_results)} results collected.")


--- examples/documents/simple.aif ---


  [1/160] section          chunks= 6  avg_tok=   20.7  self_cont= 50%  cv=0.93


  [2/160] token-budget     chunks= 1  avg_tok=  124.0  self_cont=100%  cv=0.00


  [3/160] semantic         chunks= 4  avg_tok=   31.0  self_cont= 75%  cv=0.47


  [4/160] fixed-blocks     chunks= 3  avg_tok=   41.3  self_cont= 33%  cv=0.85

--- examples/documents/wiki_article.aif ---


...

  [156/160] fixed-blocks     chunks= 1  avg_tok= 3815.0  self_cont=  0%  cv=0.00

--- examples/skills/superpowers/superpowers-writing-plans.aif ---


  [157/160] section          chunks= 1  avg_tok= 4541.0  self_cont=  0%  cv=0.00


  [158/160] token-budget     chunks= 1  avg_tok= 4541.0  self_cont=  0%  cv=0.00


  [159/160] semantic         chunks= 1  avg_tok= 4541.0  self_cont=  0%  cv=0.00


  [160/160] fixed-blocks     chunks= 1  avg_tok= 4541.0  self_cont=  0%  cv=0.00

Done. 160 results collected.


## Results Table

Aggregate metrics per strategy across all documents. The key question: which strategy produces the most **self-contained** chunks (good for LLM reasoning) with the most **uniform sizes** (good for context window packing)?

In [5]:
print(f"{'Strategy':15s} {'Avg Chunks':>11s} {'Self-Cont.':>11s} {'Size CV':>8s} {'Avg Blk/Chk':>13s}")
print("-" * 62)

strategy_summaries = {}

for strategy in STRATEGIES:
    strat_results = [r for r in all_results if r.get("strategy") == strategy and "error" not in r]
    if not strat_results:
        continue

    total_chunks = sum(r["num_chunks"] for r in strat_results)
    avg_chunks = total_chunks / len(strat_results)
    avg_sc = statistics.mean(r["self_containment"] for r in strat_results)
    avg_cv = statistics.mean(r["size_cv"] for r in strat_results)
    avg_bpc = statistics.mean(r["avg_blocks_per_chunk"] for r in strat_results)

    strategy_summaries[strategy] = {
        "avg_chunks": round(avg_chunks, 1),
        "self_containment": round(avg_sc, 3),
        "size_cv": round(avg_cv, 3),
        "avg_blocks_per_chunk": round(avg_bpc, 1),
    }

    sc_pct = f"{avg_sc:.0%}"
    print(f"{strategy:15s} {avg_chunks:11.1f} {sc_pct:>11s} {avg_cv:8.2f} {avg_bpc:13.1f}")

# Highlight best strategy for self-containment
if strategy_summaries:
    best_sc = max(strategy_summaries, key=lambda s: strategy_summaries[s]["self_containment"])
    best_cv = min(strategy_summaries, key=lambda s: strategy_summaries[s]["size_cv"])
    print(f"\nHighest self-containment: {best_sc} ({strategy_summaries[best_sc]['self_containment']:.0%})")
    print(f"Most uniform chunk sizes: {best_cv} (CV={strategy_summaries[best_cv]['size_cv']:.2f})")

Strategy         Avg Chunks  Self-Cont.  Size CV   Avg Blk/Chk
--------------------------------------------------------------
section                 1.7          4%     0.08           1.1
token-budget            1.0          8%     0.00           2.3
semantic                1.6          5%     0.04           1.1
fixed-blocks            1.2          3%     0.04           1.3

Highest self-containment: token-budget (8%)
Most uniform chunk sizes: token-budget (CV=0.00)


## Per-Document Details

Breakdown showing how each strategy performs on each individual document.

In [6]:
# Group results by file
files_seen = []
for r in all_results:
    if r["file"] not in files_seen:
        files_seen.append(r["file"])

for filename in files_seen:
    file_results = [r for r in all_results if r["file"] == filename]
    print(f"\n{'=' * 70}")
    print(f"  {filename}")
    print(f"{'=' * 70}")
    print(f"  {'Strategy':15s} {'Chunks':>7s} {'Avg Tok':>8s} {'Range':>15s} {'Self-Cont':>10s} {'CV':>6s} {'Blk/Chk':>8s}")
    print(f"  {'-' * 69}")
    for r in file_results:
        if "error" in r:
            print(f"  {r['strategy']:15s}  ERROR: {r['error']}")
            continue
        tok_range = f"[{r['min_tokens']}-{r['max_tokens']}]"
        sc_pct = f"{r['self_containment']:.0%}"
        budget_note = ""
        if r["budget_compliance"] is not None:
            budget_note = f"  budget={r['budget_compliance']:.0%}"
        print(
            f"  {r['strategy']:15s} {r['num_chunks']:7d} {r['mean_tokens']:8.1f} {tok_range:>15s} "
            f"{sc_pct:>10s} {r['size_cv']:6.2f} {r['avg_blocks_per_chunk']:8.1f}{budget_note}"
        )


  simple.aif
  Strategy         Chunks  Avg Tok           Range  Self-Cont     CV  Blk/Chk
  ---------------------------------------------------------------------
  section               6     20.7          [3-42]        50%   0.93      1.8
  token-budget          1    124.0       [124-124]       100%   0.00     11.0  budget=100%
  semantic              4     31.0         [10-43]        75%   0.47      2.8
  fixed-blocks          3     41.3          [7-77]        33%   0.85      3.7

  wiki_article.aif
  Strategy         Chunks  Avg Tok           Range  Self-Cont     CV  Blk/Chk
  ---------------------------------------------------------------------
  section              14     28.1          [2-85]        50%   1.09      1.6
  token-budget          1    394.0       [394-394]       100%   0.00     23.0  budget=100%
  semantic             11     35.8          [3-58]        64%   0.51      2.1
  fixed-blocks          5     78.8        [47-110]        60%   0.37      4.6

  migration_esl

## Save Results

In [7]:
output_path = Path(os.path.abspath("")) / "results.json"
output_data = {
    "strategy_summaries": strategy_summaries,
    "per_document": all_results,
}
with open(output_path, "w") as f:
    json.dump(output_data, f, indent=2)
print(f"Results saved to {output_path}")

Results saved to /Users/liqunchen/Claude_Project/token_efficient/benchmarks/chunking/results.json


## Findings

### Typed blocks create natural chunking boundaries

AIF's semantic blocks (`@claim`, `@evidence`, `@step`, `@verify`, `@precondition`, `@definition`)
carry explicit meaning that chunking strategies can respect. The key insight is not just chunk *size*
but chunk *coherence* --- does each chunk contain a complete semantic unit that an LLM can reason about?

**Section-based** and **semantic** chunking split at natural document boundaries (section headings
and semantic block boundaries), producing chunks where the LLM immediately knows what kind of content
it is reading. A chunk starting with `@step[order=1]` followed by `@verify` gives the model a
complete execute-then-validate unit.

### Which strategy is best for LLM understanding?

It depends on the LLM's task:

- **For instruction-following agents**: **Semantic** chunking preserves typed block boundaries.
  An agent receiving a `@step` + `@verify` pair in one chunk can execute and validate without
  cross-chunk lookups. The typed blocks tell the agent *what to do* and *how to check it*.
- **For RAG with context-window packing**: **Token Budget** guarantees no chunk exceeds the
  target, so you can pack exactly N chunks into a prompt without overflow.
- **For document comprehension**: **Section** chunking maps chunks to logical topics, helping
  the LLM produce structured summaries that respect the author's organization.
- **Fixed Blocks** is a baseline --- it ignores semantic structure and often splits mid-block,
  producing chunks the LLM struggles to interpret without surrounding context.

### Semantic boundaries improve instruction following

When a chunk boundary falls between a `@step` and its `@verify`, the LLM loses the ability to
check its own work. When it falls between a `@claim` and its `@evidence`, the LLM cannot distinguish
assertion from support. Semantic and section-based strategies avoid this by keeping related typed
blocks together, directly improving the model's ability to understand document structure and follow
multi-step instructions.

This is the fundamental advantage of AIF over raw text for chunking: **typed blocks make chunk
boundaries meaningful**, not arbitrary.